## Installing Dependencies
---

1. `pciutils` is required by Ollama to detect the GPU type.
2. Installation of Ollama in the runtime instance will be taken care by `curl -fsSL https://ollama.com/install.sh | sh`




In [ ]:
!sudo apt update
!sudo apt install -y pciutils zstd
!curl -fsSL https://ollama.com/install.sh | sh

## Running Ollama
---

In order to use Ollama it needs to run as a service in background parallel to your scripts. Becasue Jupyter Notebooks is built to run code blocks in sequence this make it difficult to run two blocks at the same time. As a workaround we will create a service using subprocess in Python so it doesn't block any cell from running.

Service can be started by command `ollama serve`.

`time.sleep(5)` adds some delay to get the Ollama service up before downloading the model.

In [ ]:
import threading
import subprocess
import time

def run_ollama_serve():
  subprocess.Popen(["ollama", "serve"])

thread = threading.Thread(target=run_ollama_serve)
thread.start()
time.sleep(5)

## Pulling Model
---

Download the LLM model using `ollama pull llama3.2`.

For other models check https://ollama.com/library

In [ ]:
!ollama pull bge-m3

In [ ]:
!pip install -U langchain langchain-core langchain-community


In [ ]:
!pip install -U langchain-ollama

## Embedding Text with Ollama

Now that Ollama is running and the `bge-m3` model is pulled, we can use `langchain-ollama` to create embeddings for your text data.

First, we'll load your JSON file. Please ensure `direct_taxes_law_v1.json` is available in your environment, for example, by uploading it to your Colab session or mounting Google Drive.

In [ ]:
import json

# Load the JSON data from the file
try:
    with open('direct_taxes_law_v1.json', 'r') as f:
        data = json.load(f)
    print(f"Successfully loaded {len(data)} items from direct_taxes_law_v1.json")
except FileNotFoundError:
    print("Error: 'direct_taxes_law_v1.json' not found. Please upload the file or provide the correct path.")
    data = [] # Initialize data as empty list to prevent further errors
except json.JSONDecodeError:
    print("Error: Could not decode JSON from 'direct_taxes_law_v1.json'. Please check file content.")
    data = []

Next, we initialize the `OllamaEmbeddings` using the `bge-m3` model. Note that the `base_url` is set to the default Ollama API address.

In [ ]:
from langchain_community.embeddings import OllamaEmbeddings

# Initialize Ollama embeddings with the bge-m3 model
embeddings = OllamaEmbeddings(model="bge-m3", base_url="http://localhost:11434")
print("OllamaEmbeddings initialized with model: bge-m3")

In [ ]:
!pip install tqdm

In [ ]:
import tqdm

output = []
errors = []
with tqdm.tqdm(total=len(data), desc='Embedding laws') as pbar:
    for item in data:
        try:
          vector = embeddings.embed_query(item["finalSectionText"])
        except ValueError as e:
          print(f"Error embedding text: {e}")
          errors.append(item["sectionId"])
          continue
        output.append(
            {
              "id": item["sectionId"],
              "vector": vector,
              "text": item["finalSectionText"],
              "metadata": {k: v for k, v in item.items() if k != "finalSectionText"}
            }
        )
        pbar.update(1)
print("errors:", errors)

In [ ]:
with open('output.json', 'w') as f:
    json.dump(output, f, indent=2)